# Step 1 — Preprocessing Teks

**Tujuan praktikum**
1. Mahasiswa memahami alur preprocessing teks: *case folding*, tokenisasi, penghapusan *stopword*, dan *stemming*.
2. Mahasiswa mampu mengimplementasikan preprocessing untuk teks berbahasa Indonesia dengan pustaka **Sastrawi**.
3. Mahasiswa mampu mengukur pengaruh tiap tahap preprocessing terhadap jumlah token dan ukuran kosakata.

Korpus praktikum berada di `../data/korpus.csv` dan berisi 30 dokumen berita pendek berbahasa Indonesia dari lima topik: kesehatan, pertanian, teknologi informasi, ekonomi, dan pendidikan.

In [ ]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## Langkah 1 — Memuat korpus

In [ ]:
korpus = pd.read_csv("../data/korpus.csv")
print("Jumlah dokumen:", len(korpus))
korpus.head()

In [ ]:
# Judul dan isi digabung menjadi satu teks dokumen
korpus["teks"] = korpus["judul"] + ". " + korpus["isi"]
contoh = korpus.loc[16, "teks"]
print(contoh)

## Langkah 2 — Case folding
*Case folding* menyeragamkan huruf menjadi huruf kecil sehingga `Stemming`, `stemming`, dan `STEMMING` dianggap kata yang sama.

In [ ]:
def case_folding(teks):
    return teks.lower()

print(case_folding(contoh))

## Langkah 3 — Tokenisasi
Tokenisasi memecah teks menjadi satuan kata (token). Di sini digunakan *regular expression* `[a-z0-9]+`: setiap rangkaian huruf/angka dianggap satu token, sedangkan tanda baca menjadi pemisah.

In [ ]:
import re

def tokenize(teks):
    return re.findall(r"[a-z0-9]+", teks.lower())

token_contoh = tokenize(case_folding(contoh))
print(len(token_contoh), "token")
print(token_contoh)

In [ ]:
# Bandingkan dengan tokenisasi naif menggunakan split() — tanda baca masih menempel
print(case_folding(contoh).split()[:12])

## Langkah 4 — Penghapusan stopword
*Stopword* adalah kata yang sangat sering muncul tetapi hampir tidak membawa makna topik, misalnya *yang, dan, di, untuk*. Sastrawi menyediakan daftar stopword bahasa Indonesia.

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
print("Jumlah stopword:", len(STOPWORDS))
print(sorted(STOPWORDS)[:40])

In [ ]:
def remove_stopwords(tokens, stopwords=STOPWORDS):
    return [t for t in tokens if t not in stopwords]

tanpa_sw = remove_stopwords(token_contoh)
print(len(token_contoh), "->", len(tanpa_sw), "token")
print(tanpa_sw)

## Langkah 5 — Stemming
*Stemming* mengubah kata berimbuhan menjadi kata dasar. Bahasa Indonesia kaya imbuhan (awalan, akhiran, sisipan, konfiks), sehingga Sastrawi memakai algoritma berbasis kamus kata dasar (Nazief–Adriani yang diperluas).

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
for kata in ["pembelajaran", "pertanian", "petani", "pencarian", "keamanan",
             "memantau", "berolahraga", "perekonomian", "mahasiswa"]:
    print(f"{kata:15s} -> {stemmer.stem(kata)}")

Stemming Sastrawi relatif lambat karena setiap kata dicocokkan ke kamus. Karena kata yang sama muncul berulang kali, hasilnya disimpan dalam *cache* (dictionary) agar setiap kata unik hanya di-stem sekali.

In [ ]:
_cache = {}

def stem(token):
    if token not in _cache:
        _cache[token] = stemmer.stem(token)
    return _cache[token]

print([stem(t) for t in tanpa_sw])

## Langkah 6 — Menggabungkan menjadi satu fungsi `preprocess`

In [ ]:
def preprocess(teks, use_stopword=True, use_stemming=True):
    tokens = tokenize(case_folding(teks))
    if use_stopword:
        tokens = remove_stopwords(tokens)
    if use_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens

korpus["tokens"] = korpus["teks"].apply(preprocess)
korpus[["doc_id", "judul", "tokens"]].head()

## Langkah 7 — Mengukur pengaruh tiap tahap

In [ ]:
tahap = {
    "Tokenisasi saja": dict(use_stopword=False, use_stemming=False),
    "+ Stopword removal": dict(use_stopword=True, use_stemming=False),
    "+ Stemming (tanpa stopword removal)": dict(use_stopword=False, use_stemming=True),
    "+ Stopword removal + Stemming": dict(use_stopword=True, use_stemming=True),
}
baris = []
for nama, opsi in tahap.items():
    semua = [t for teks in korpus["teks"] for t in preprocess(teks, **opsi)]
    baris.append({"Tahap": nama, "Jumlah token": len(semua), "Ukuran kosakata": len(set(semua))})
pd.DataFrame(baris)

In [ ]:
from collections import Counter

frekuensi = Counter(t for tokens in korpus["tokens"] for t in tokens)
pd.DataFrame(frekuensi.most_common(15), columns=["term", "frekuensi"])

## Langkah 8 — Menyimpan fungsi untuk step berikutnya
Fungsi yang sama sudah disediakan di `stki_utils.py` agar notebook step berikutnya cukup mengimpornya. Sel berikut memastikan hasil fungsi Anda identik dengan versi di `stki_utils.py`.

In [ ]:
import stki_utils

cocok = all(preprocess(t) == stki_utils.preprocess(t) for t in korpus["teks"])
print("Hasil identik dengan stki_utils.preprocess:", cocok)

## Tugas Step 1
1. Temukan minimal **tiga** kasus *overstemming* atau *understemming* Sastrawi pada korpus (misalnya dua kata berbeda makna yang menghasilkan stem sama). Jelaskan dampaknya terhadap pencarian.
2. Tambahkan stopword khusus domain (misalnya `dapat`, `membantu`) ke daftar stopword. Bandingkan ukuran kosakata sebelum dan sesudahnya.
3. Ubah tokenizer agar kata berulang seperti `anak-anak` atau `sayur-mayur` tetap menjadi satu token. Uji pada kalimat buatan Anda sendiri.
4. Tuliskan analisis singkat (maksimal 1 paragraf): tahap mana yang paling banyak mengurangi ukuran kosakata, dan mengapa?

In [ ]:
# Tulis jawaban Anda di sini